 # Artificial Intelligence Technology and Application

 ## Machine Learning Lab Guide - Student Version

 # 1 Private Credit Default Prediction

 ## 1.1 Introduction

 This experiment implements private credit default prediction from scratch, helping identify high-risk customers efficiently.

 ## 1.2 Procedure

 ### 1.2.1 Reading Data

In [1]:
import pandas as pd
import numpy as np

# Constructing mock dataset representing credit defaults
# Features: Income, Age, LoanAmount, Duration, Target(Default=1)
np.random.seed(42)
n_samples = 1000

# Creating an imbalanced dataset: 5% default rate
target = np.random.choice([0, 1], p=[0.95, 0.05], size=n_samples)
income = np.random.normal(50000, 15000, n_samples)
age = np.random.randint(20, 70, n_samples).astype(float)
loan_amt = np.random.normal(10000, 5000, n_samples)
duration = np.random.choice([12, 24, 36, 48, 60], size=n_samples)

# Adding missing values randomly to age and loan_amt to match lab logic
age[np.random.choice(n_samples, 50, replace=False)] = np.nan
loan_amt[np.random.choice(n_samples, 80, replace=False)] = np.nan

df = pd.DataFrame({
    'Income': income,
    'Age': age,
    'LoanAmount': loan_amt,
    'Duration': duration,
    'Target': target
})

print("First 5 rows of the dataset:")
print(df.head())


First 5 rows of the dataset:
         Income   Age    LoanAmount  Duration  Target
0  52665.515014  20.0  14317.983296        12       0
1  29969.834619  48.0  12898.216147        12       1
2  55702.967765  64.0   7698.682992        24       0
3  59158.786179  68.0   4322.695532        60       0
4  58396.856719  54.0  17298.659375        48       0


 ### 1.2.2 Viewing Missing Values

In [2]:
print("\nChecking for missing values:")
print(df.isnull().sum())

# Fill missing values with the mode as specified by the lab document
missing_cols = ['Age', 'LoanAmount']
for col in missing_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

print("\nMissing values after filling:")
print(df.isnull().sum())



Checking for missing values:
Income         0
Age           50
LoanAmount    80
Duration       0
Target         0
dtype: int64

Missing values after filling:
Income        0
Age           0
LoanAmount    0
Duration      0
Target        0
dtype: int64


 ### 1.2.3 Splitting the Dataset

 Before splitting, extract target y and features X.

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('Target', axis=1)
y = df['Target']

# Splitting dataset into 90% training and 10% testing. shuffle=True is default.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)



Training set shape: (900, 4)
Testing set shape: (100, 4)


 ### 1.2.4 Standardizing Data (Preprocessing Data)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nX_train standardized (first row):")
print(X_train_scaled[0])



X_train standardized (first row):
[-0.72680158 -0.55824793  0.29550973 -0.65598228]


 ### 1.2.5 Handling the Class Imbalance Issue (Preprocessing)

 Displaying the class ratio, and applying SMOTE (Synthetic Minority Over-sampling Technique) to handle imbalance.

In [5]:
print("\nClass distribution in Training set:")
print(y_train.value_counts(normalize=True))

try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    print("\nClass distribution after SMOTE:")
    print(y_train_res.value_counts())
except ImportError:
    print("\n'imbalanced-learn' is not installed. Proceeding without SMOTE.")
    print("Use `pip install imbalanced-learn` to enable SMOTE oversampling.")
    X_train_res, y_train_res = X_train_scaled, y_train



Class distribution in Training set:
Target
0    0.954444
1    0.045556
Name: proportion, dtype: float64

'imbalanced-learn' is not installed. Proceeding without SMOTE.
Use `pip install imbalanced-learn` to enable SMOTE oversampling.


 ### 1.2.6 Performing Grid Search (Modeling)

 Logistic Regression hyperparameter tuning using GridSearchCV.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Defining parameter grid 
param_grid = {
    'C': [0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'liblinear']
}

lr = LogisticRegression(max_iter=1000)
grid_search = GridSearchCV(lr, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_res, y_train_res)

print("\nBest Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

# Evaluating optimal model
best_model = grid_search.best_estimator_



Best Parameters: {'C': 0.1, 'solver': 'lbfgs'}
Best CV Score: 0.9544444444444444


 ### 1.2.7 Verifying Performance (Evaluation)

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Predictions on Train set (for reference)
y_train_pred = best_model.predict(X_train_res)
print("\n--- Training Set Metrics ---")
print("Accuracy score is:", accuracy_score(y_train_res, y_train_pred))
print("Precision score is:", precision_score(y_train_res, y_train_pred))
print("Recall score is:", recall_score(y_train_res, y_train_pred))
print("AUC score is:", roc_auc_score(y_train_res, y_train_pred))

# Predictions on unseen Test set
y_test_pred = best_model.predict(X_test_scaled)
print("\n--- Testing Set Metrics ---")
print("Accuracy score is:", accuracy_score(y_test, y_test_pred))
print("Precision score is:", precision_score(y_test, y_test_pred, zero_division=0))
print("Recall score is:", recall_score(y_test, y_test_pred))
print("AUC score is:", roc_auc_score(y_test, y_test_pred))




--- Training Set Metrics ---
Accuracy score is: 0.9544444444444444
Precision score is: 0.0
Recall score is: 0.0
AUC score is: 0.5

--- Testing Set Metrics ---
Accuracy score is: 0.95
Precision score is: 0.0
Recall score is: 0.0
AUC score is: 0.5


/home/titan/.cache/uv/archive-v0/WbN29UG76Z7ZMefV38NKD/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


 ### 1.2.8 Saving the Model

In [8]:
import joblib

model_filename = "credit_default_model.pkl"
joblib.dump(best_model, model_filename)
print(f"\nModel saved successfully as: {model_filename}")

# To load the model later:
# loaded_model = joblib.load(model_filename)



Model saved successfully as: credit_default_model.pkl
